# ComicBookGenerator model test

This notebook runs the same local panel renderer used by the API. Select `sd15` or `sdxl_dreamshaper`; the SDXL checkpoint is downloaded only when that option is selected.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Enable a GPU runtime in Colab.')

In [ ]:
# Upload/clone the repository before running this notebook.
# Example:
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git /content/ComicBookGenerator
%cd /content/ComicBookGenerator
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
# Optional: keep the SD 1.5 checkpoint/LoRAs in Drive.
# !mkdir -p models
# !ln -s /content/drive/MyDrive/ComicBookGenerator/models/base models/base
# !ln -s /content/drive/MyDrive/ComicBookGenerator/models/loras models/loras

In [ ]:
MODEL = 'sdxl_dreamshaper'  # change to 'sd15' to test the local checkpoint
PROMPT = 'A short four-panel comic about a sleepy robot delivering one parcel, clear setup and visual punchline, clean comic illustration.'
import json
from core.model_registry import ensure_model
profile, model_path = ensure_model(MODEL)
print(profile['label'], model_path)

In [ ]:
from diffusers import StableDiffusionXLPipeline
from PIL import Image
if MODEL == 'sdxl_dreamshaper':
    pipe = StableDiffusionXLPipeline.from_pretrained(model_path, torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')
    image = pipe(PROMPT, negative_prompt='blurry, extra characters, watermark, text', width=768, height=768, num_inference_steps=20, guidance_scale=6.5).images[0]
    out = Path('outputs/colab_sdxl_smoke.png')
    out.parent.mkdir(exist_ok=True)
    image.save(out)
    display(image)
else:
    print('SD 1.5 model resolved. Use the API or targeted_smoke.py for the full schema/page workflow.')

## Full API workflow (optional)

The next cells start the same FastAPI graph used by the local UI. Set `MODEL` to `sd15` or `sdxl_dreamshaper`; the page and job files are written under `outputs/`.

In [ ]:
import subprocess, time, requests, json
api_process = subprocess.Popen(['python', 'api.py'], cwd='/content/ComicBookGenerator')
time.sleep(20)
print(requests.get('http://127.0.0.1:8000/api/capabilities').json())

In [ ]:
payload = {
    'prompt': 'Write a short four-panel comic in natural English about a careful robot delivering one parcel, with a clear setup, one complication, and a visual punchline. Keep one recurring robot and one parcel.',
    'model': MODEL, 'steps': 20, 'guidance': 7.0, 'lora': '', 'seed': '12345', 'pageCount': 1
}
events = []
with requests.post('http://127.0.0.1:8000/api/generate', json=payload, stream=True, timeout=1800) as response:
    response.raise_for_status()
    for line in response.iter_lines(decode_unicode=True):
        if line and line.startswith('data: '):
            event = line[6:]
            events.append(event)
            print(event)
Path('outputs/colab_events.json').write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding='utf-8')